In [2]:
import json
import pandas as pd

def extract_mondo_diseases(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    nodes = data['graphs'][0]['nodes']
    disease_list = []
    
    for node in nodes:
        # IDの取得と整形
        raw_id = node.get('id', '')
        
        # Mondo疾患ノードは通常 "MONDO_" という文字列を含み、かつタイプが "CLASS" である
        if 'MONDO_' not in raw_id or node.get('type') != 'CLASS':
            continue
            
        # IDを "MONDO:0008170" の形式に整形
        mondo_id = raw_id.split('/')[-1].replace('_', ':')
        
        # 疾患名 (label)
        label = node.get('lbl', '')
        
        # メタデータ (シノニムと外部参照)
        meta = node.get('meta', {})
        
        # シノニム: 特徴量A (PubMed) の検索ワードに使用
        synonyms = [s['val'] for s in meta.get('synonyms', [])]
        
        # 外部参照: GlobalData等のラベル紐付けに使用 (ICD10, Orphanet, OMIM等)
        xrefs = [x['val'] for x in meta.get('xrefs', [])]
        
        disease_list.append({
            'mondo_id': mondo_id,
            'label': label,
            'synonyms': synonyms,
            'xrefs': xrefs
        })
    
    # DataFrame化
    df = pd.DataFrame(disease_list)
    
    # 重複削除とソート
    df = df.drop_duplicates(subset=['mondo_id']).sort_values('mondo_id')
    
    return df

# 実行
df_master = extract_mondo_diseases('/Users/user/dev/projects/Mondo/mondo.json')
# 「obsolete」を含まないものだけを残す
df_master = df_master[~df_master['label'].str.contains('obsolete', case=False)]
print(df_master.head())

         mondo_id                                       label  \
0   MONDO:0000001                                     disease   
3   MONDO:0000004                adrenocortical insufficiency   
4   MONDO:0000005                          alopecia, isolated   
8   MONDO:0000009  inherited bleeding disorder, platelet-type   
13  MONDO:0000014                     colorblindness, partial   

                                             synonyms  \
0   [condition, disease, disease or disorder, dise...   
3   [adrenal cortical hypofunction, adrenal cortic...   
4                                                  []   
8   [blood platelet disease, platelet disorder, bl...   
13                                                 []   

                                                xrefs  
0   [DOID:4, ICD9:799.9, MESH:D004194, NCIT:C2991,...  
3   [DOID:10493, EFO:0009491, ICD9:255.4, ICD9:255...  
4                                     [OMIMPS:203655]  
8   [DOID:2218, GARD:0022702, MEDGEN:610, 

In [3]:
import pandas as pd
import time
import os
import requests
from Bio import Entrez
from tqdm import tqdm

# PubMed設定
Entrez.email = "m.keigo.sp.ku@gmail.com" 

def get_pubmed_count(query):
    try:
        handle = Entrez.esearch(db="pubmed", term=query, rettype="count")
        record = Entrez.read(handle)
        handle.close()
        return int(record["Count"])
    except:
        return 0

def get_clinical_trials_data(query):
    base_url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "query.term": query,
        "filter.overallStatus": "RECRUITING|ENROLLING_BY_INVITATION",
        "pageSize": 1000
    }
    try:
        response = requests.get(base_url, params=params, timeout=10)
        data = response.json()
        studies = data.get('studies', [])
        trial_count = len(studies)
        total_enrollment = sum(
            int(s.get('protocolSection', {}).get('designModule', {}).get('enrollmentInfo', {}).get('count', 0))
            for s in studies
        )
        return trial_count, total_enrollment
    except:
        return 0, 0

def process_features_with_progress(df_master, filename="mondo_activity_features.csv"):
    # 1. 既存データの読み込み（再開用）
    if os.path.exists(filename):
        df_progress = pd.read_csv(filename)
        processed_ids = set(df_progress['mondo_id'].unique())
        print(f"再開モード: {len(processed_ids)} 件の処理済みデータが見つかりました。")
    else:
        df_progress = pd.DataFrame()
        processed_ids = set()

    # 未処理分を抽出
    df_todo = df_master[~df_master['mondo_id'].isin(processed_ids)]
    
    if df_todo.empty:
        print("すべてのデータは処理済みです。")
        return df_progress

    # 2. tqdmによる進捗バー付きループ
    results = []
    # 辞書形式で既存の結果を保持
    final_results = df_progress.to_dict('records') if not df_progress.empty else []

    try:
        # totalは未処理件数
        for _, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Mondoデータ取得中"):
            label = row['label']
            
            pm_count = get_pubmed_count(label)
            ct_count, ct_enrollment = get_clinical_trials_data(label)
            
            new_record = {
                'mondo_id': row['mondo_id'],
                'label': label,
                'pubmed_total_count': pm_count,
                'ct_active_trials': ct_count,
                'ct_total_enrollment_target': ct_enrollment
            }
            final_results.append(new_record)
            
            # 定期的に保存（例：50件ごと）
            if len(final_results) % 50 == 0:
                pd.DataFrame(final_results).to_csv(filename, index=False)
            
            time.sleep(0.4) # API制限対策

    except KeyboardInterrupt:
        print("\n中断されました。現在の進捗を保存します...")
    finally:
        # 最終保存
        df_final = pd.DataFrame(final_results)
        df_final.to_csv(filename, index=False)
        print(f"保存完了: {filename}")

    return df_final

# 実行
df_features = process_features_with_progress(df_master)

Mondoデータ取得中: 100%|██████████| 28064/28064 [13:31:29<00:00,  1.73s/it]  

保存完了: mondo_activity_features.csv


In [4]:
import pandas as pd
import requests
import time
import os
from tqdm import tqdm
from urllib.parse import quote

def get_digital_influence_metrics(disease_label):
    headers = {'User-Agent': 'MondoDiseaseAnalysis/1.1 (contact: your_email@example.com)'}
    
    # デフォルト値
    wiki_pv = 0
    wiki_langs = 0
    google_hits = 0

    try:
        # 1. Wikipedia 記事検索と言語リンク数の取得
        search_url = "https://en.wikipedia.org/w/api.php"
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": disease_label,
            "format": "json"
        }
        search_res = requests.get(search_url, params=search_params, headers=headers, timeout=5).json()
        
        if search_res.get('query', {}).get('search'):
            page_title = search_res['query']['search'][0]['title']
            page_title_encoded = quote(page_title.replace(' ', '_'))

            # 1-a. Pageviews取得 (2025年の1年間分)
            pv_url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/all-agents/{page_title_encoded}/monthly/20250101/20251231"
            pv_res = requests.get(pv_url, headers=headers, timeout=5).json()
            wiki_pv = sum(item['views'] for item in pv_res.get('items', []))

            # 1-b. 言語リンク数取得
            lang_params = {
                "action": "query",
                "prop": "langlinks",
                "titles": page_title,
                "format": "json",
                "lllimit": 500
            }
            lang_res = requests.get(search_url, params=lang_params, headers=headers, timeout=5).json()
            pages = lang_res.get('query', {}).get('pages', {})
            page_id = list(pages.keys())[0]
            wiki_langs = len(pages[page_id].get('langlinks', [])) + 1 # +1は英語版

        # 2. Google Search "Result Count" (SerpApi等を使わない簡易推定法)
        # 厳密な数値より「桁数」が重要。ここではCustom Search APIか簡易スクレイピングを検討。
        # 注意: 短時間の大量リクエストはブロックされるため、慎重にスリープを入れます。
        # 以下はロジックのプレースホルダー（実際の運用では低速実行必須）
        google_hits = 0 # Googleは特に制限が厳しいため、最初は0で設定しWikipediaを優先。
        
    except Exception as e:
        pass # エラー時は0のまま返す
        
    return wiki_pv, wiki_langs, google_hits

def enrich_with_digital_influence(input_file="mondo_activity_features.csv", output_file="mondo_digital_influence.csv"):
    # 既存の進行状況を確認
    if os.path.exists(output_file):
        df_final = pd.read_csv(output_file)
        processed_ids = set(df_final['mondo_id'].unique())
        print(f"再開: {len(processed_ids)} 件処理済み")
    else:
        df_master = pd.read_csv(input_file)
        # 必要なカラムを初期化
        df_master['wiki_annual_pv'] = 0
        df_master['wiki_lang_count'] = 0
        df_master['google_result_count'] = 0
        df_final = df_master.copy()
        processed_ids = set()

    # 未処理の行を特定
    todo_mask = ~df_final['mondo_id'].isin(processed_ids)
    df_todo = df_final[todo_mask]

    try:
        for index, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc="Digital Influence取得中"):
            pv, langs, hits = get_digital_influence_metrics(row['label'])
            
            df_final.at[index, 'wiki_annual_pv'] = pv
            df_final.at[index, 'wiki_lang_count'] = langs
            df_final.at[index, 'google_result_count'] = hits
            
            # 50件ごとに保存
            if index % 50 == 0:
                df_final.to_csv(output_file, index=False)
            
            time.sleep(0.5) # レート制限対策

    except KeyboardInterrupt:
        print("\n中断されました。保存します...")
    finally:
        df_final.to_csv(output_file, index=False)
        print(f"最終保存完了: {output_file}")

    return df_final

# 実行
df_digital = enrich_with_digital_influence()

Digital Influence取得中: 100%|██████████| 28064/28064 [18:30:15<00:00,  2.37s/it]   

最終保存完了: mondo_digital_influence.csv


In [5]:
import json
import pandas as pd
import networkx as nx
from tqdm import tqdm

def extract_hierarchical_features(json_path):
    print(f"Loading {json_path}...")
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    graph_data = data['graphs'][0]
    nodes = graph_data['nodes']
    edges = graph_data['edges']
    
    # 1. グラフの構築 (DiGraph: 有向グラフ)
    G = nx.DiGraph()
    
    # 疾患ノード（Mondo IDを持つCLASSタイプ）を追加
    for node in nodes:
        node_id = node.get('id', '')
        if 'MONDO_' in node_id and node.get('type') == 'CLASS':
            # IDを "MONDO:XXXXXXX" 形式に整形
            mondo_id = node_id.split('/')[-1].replace('_', ':')
            G.add_node(mondo_id)
            
    # 親子関係（is_a）のエッジを追加
    for edge in edges:
        # predが 'is_a'（subClassOf）の関係のみ抽出
        if edge.get('pred') == 'is_a':
            parent = edge.get('obj', '').split('/')[-1].replace('_', ':')
            child = edge.get('sub', '').split('/')[-1].replace('_', ':')
            # 両方のノードがグラフに存在する場合のみ追加
            if parent in G and child in G:
                G.add_edge(parent, child)

    # 2. 特徴量の計算
    # ルートノード（MONDO:0000001: disease）からの最短経路を計算
    root = "MONDO:0000001"
    
    # ルートが存在しない場合のエラー回避
    if root not in G:
        print(f"Warning: Root node {root} not found in the graph.")
        # 便宜上、入次数0のノードをルート候補とするなどの処理が必要
    
    # 全ノードへの最短経路長（深さ）を取得
    depths = nx.single_source_shortest_path_length(G, root)
    
    hierarchical_data = []
    
    for node in tqdm(G.nodes(), desc="Analyzing hierarchy"):
        # 特徴量1: 深さ (ルートから辿れない場合は -1 または 0)
        depth = depths.get(node, 0)
        
        # 特徴量2: 子ノードの数（その疾患がどれだけ細分化されているか）
        children = list(G.successors(node))
        child_count = len(children)
        
        # 特徴量3: リーフノード判定（これ以上細分化されない具体的疾患か）
        is_leaf = 1 if child_count == 0 else 0
        
        hierarchical_data.append({
            'mondo_id': node,
            'mondo_depth': depth,
            'mondo_child_count': child_count,
            'mondo_is_leaf': is_leaf
        })
        
    return pd.DataFrame(hierarchical_data)

# --- 実行と統合の手順 ---
# 1. 階層データの作成
df_hierarchy = extract_hierarchical_features('mondo.json')

# 2. 既存のデジタル指標データ（前回作成したもの）とマージ
df_digital = pd.read_csv('mondo_digital_influence.csv')
df_final = pd.merge(df_digital, df_hierarchy, on='mondo_id', how='left')

# 3. 最終保存
df_final.to_csv('mondo_full_features.csv', index=False)

Loading mondo.json...


Analyzing hierarchy: 100%|██████████| 31958/31958 [00:00<00:00, 1630180.20it/s]
